# EnsembleLab generation and optimization demo

This notebook uses the current package API. Run it from an environment where `ensemblelab` is installed, or install the project in editable mode with `pip install -e .`.

In [3]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "src").is_dir():
    project_root = project_root.parent

if not (project_root / "src" / "ensemblelab").is_dir():
    raise RuntimeError("Open this notebook from within the ensemblelab repository.")

sys.path.insert(0, str(project_root / "src"))

from ensemblelab import generate, hierarchical_optimize, optimize
from ensemblelab.optimizers import MMFFOptimizer


## Generate

`generate()` returns an unoptimized `Ensemble`. Its conformers have aligned RDKit and ASE geometries, but no energies yet.

In [4]:
ensemble = generate("CCO", n_confs=10)

print(f"Generated conformers: {len(ensemble.conformers)}")
print(f"Conformer IDs: {ensemble.conformer_ids}")
print(f"Energy status: {ensemble.metadata['energy_status']}")
assert all(conformer.energy is None for conformer in ensemble.conformers)


Generated conformers: 10
Conformer IDs: (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
Energy status: uncomputed


## Optimize with the current function API

MMFF is fully local through RDKit, so it is the recommended first smoke test. `optimize()` returns a new ensemble and never changes `ensemble`.

In [5]:
mmff_ensemble = optimize(ensemble, method="MMFF", max_steps=500)

assert mmff_ensemble is not ensemble
assert all(conformer.energy is None for conformer in ensemble.conformers)
assert all(conformer.energy is not None for conformer in mmff_ensemble.conformers)
assert all(conformer.energy_unit == "kcal/mol" for conformer in mmff_ensemble.conformers)

for conformer in sorted(mmff_ensemble.conformers, key=lambda item: item.energy):
    print(f"ID {conformer.id:>2}: {conformer.energy:8.3f} kcal/mol")

print(mmff_ensemble.metadata['optimization_history'][-1])


ID  7:   -1.517 kcal/mol
ID  0:   -1.337 kcal/mol
ID  4:   -1.337 kcal/mol
ID  3:   -1.337 kcal/mol
ID  2:   -1.337 kcal/mol
ID  6:   -1.337 kcal/mol
ID  9:   -1.337 kcal/mol
ID  1:   -1.337 kcal/mol
ID  8:   -1.337 kcal/mol
ID  5:   -1.337 kcal/mol
{'method': 'MMFF', 'fmax_eV_per_angstrom': None, 'max_steps': 500, 'solvent': None, 'charge': None, 'multiplicity': None, 'orca_simple_input': None, 'orca_blocks': None, 'energy_unit': 'kcal/mol', 'converged_conformer_ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'unconverged_conformer_ids': []}


## Optimizer-class API and hierarchical workflow

Optimizer classes are imported from `ensemblelab.optimizers`. A one-stage hierarchical workflow is a dependency-free way to exercise the new optimizer interface and conformer trimming.

In [6]:
optimizer = MMFFOptimizer(max_steps=500)
hierarchical_ensemble = hierarchical_optimize(
    ensemble,
    workflow=[optimizer],
    target_size=5,
    retention_rates=[],
)

assert len(hierarchical_ensemble.conformers) == 5
assert all(conformer.energy is None for conformer in ensemble.conformers)

print(hierarchical_ensemble.metadata['hierarchical_optimization'])
for conformer in hierarchical_ensemble.conformers:
    print(f"ID {conformer.id:>2}: {conformer.energy:8.3f} kcal/mol")


{'workflow': ['MMFF'], 'target_size': 5, 'retention_rates': [], 'stages': [{'optimizer': 'MMFF', 'input_size': 10, 'output_size': 5, 'minimum_energy': -1.517097576376035, 'maximum_energy': -1.3368570615014994, 'retained_energy_cutoff': -1.3368570629213692, 'energy_unit': 'kcal/mol'}]}
ID  7:   -1.517 kcal/mol
ID  0:   -1.337 kcal/mol
ID  4:   -1.337 kcal/mol
ID  3:   -1.337 kcal/mol
ID  2:   -1.337 kcal/mol


## Optional GFN2-xTB stage

After installing the optional `tblite` dependency, import `GFN2xTBOptimizer` from `ensemblelab.optimizers` and use it after MMFF. This cell is intentionally left as an example, not a default smoke test.

```python
from ensemblelab.optimizers import GFN2xTBOptimizer

refined = hierarchical_optimize(
    ensemble,
    workflow=[MMFFOptimizer(), GFN2xTBOptimizer()],
    target_size=5,
    retention_rates=[0.5],
)
```